# Naming sanitization
Notebook to sanitize all names for packages, rasters, etc.

## Modules

In [11]:
import pandas as pd
import geopandas as gpd
import fiona
import pyogrio

In [1]:
geom_layer = "geometry_layer"

In [41]:
rename_cols = {
    'cf': 'leaf', 
    'cf_median': 'leaf_median', 
    'cf_std': 'leaf_std', 
}

rename_metrics = {
    'cf_mean': 'leaf_mean', 
    'cf_median': 'leaf_median', 
    'cf_std': 'leaf_std', 
}

In [12]:
def inspect_gpkg(filepath: str):
    layer_info = pyogrio.list_layers(filepath)  # array of [name, geom_type]

    for name, geom_type in layer_info:
        if geom_type is None:
            # non-spatial table -> read without geometry
            df = gpd.read_file(filepath, layer=name, ignore_geometry=True)
            print(f"{name}: TABLE, shape={df.shape}, columns={list(df.columns)}")
        else:
            gdf = gpd.read_file(filepath, layer=name)
            print(f"{name}: {gdf.shape}, CRS={gdf.crs}, geom={gdf.geom_type.unique()}, geom_type_reported={geom_type}")

In [25]:
def load_gpkg(filepath: str):
    layer_info = pyogrio.list_layers(filepath)
    
    data = {}
    
    for name, geom_type in layer_info:
        if geom_type is None:
            df = gpd.read_file(filepath, layer=name, ignore_geometry=True)
        else:
            df = gpd.read_file(filepath, layer=name)
        data[name] = df
    
    return data

In [31]:
def update_gpkg_to_disk(data: gpd.GeoDataFrame, gpkg_path: str):
    first = True
    for name, df in data.items():
        mode = "w" if first else "a"
        if isinstance(df, gpd.GeoDataFrame):
            df.to_file(gpkg_path, layer=name, driver="GPKG", mode=mode)
        else:
            # non-spatial table: needs pyogrio directly since to_file expects geometry
            pyogrio.write_dataframe(df, gpkg_path, layer=name, driver="GPKG", append=(mode == "a"))
        first = False

    print(f'Geopackage updated into: {gpkg_path}')

## Acidification

### Ecoregion

In [50]:
acid_er_fp = "../LEAFs/acidification/acidification_ecoregion.gpkg"
acid_er_df_fp = "../LEAFs/acidification/acidification_ecoregion.csv"

In [51]:
inspect_gpkg(acid_er_fp)

geometry_layer: (829, 8), CRS=EPSG:6933, geom=['MultiPolygon'], geom_type_reported=MultiPolygon
acid_leaf_ecoregions: TABLE, shape=(4974, 6), columns=['ECO_ID', 'flow_name', 'cf', 'cf_median', 'cf_std', '_source_file']
acid_leaf_ecoregions_metadata: TABLE, shape=(6, 4), columns=['flow_name', 'impact_category', 'unit', 'source_file']


In [52]:
ac_er_gpkg_data = load_gpkg(acid_er_fp)

In [53]:
ac_er_gpkg_data['acid_leaf_ecoregions'] = ac_er_gpkg_data['acid_leaf_ecoregions'].rename(columns = rename_cols)

In [54]:
update_gpkg_to_disk(ac_er_gpkg_data, acid_er_fp)

Geopackage updated into: ../LEAFs/acidification/acidification_ecoregion.gpkg


#### df

In [55]:
acid_er_df = pd.read_csv(acid_er_df_fp)

In [56]:
acid_er_df.head()

,ECO_ID,flow_name,metric,value,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM
0,135,acid_nox,cf_mean,0.017107,Admiralty Islands lowland rain forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Australasia
1,785,acid_nox,cf_mean,0.438687,Aegean and Western Turkey sclerophyllous and m...,12.0,"Mediterranean Forests, Woodlands & Scrub",Palearctic
2,807,acid_nox,cf_mean,1.092774,Afghan Mountains semi-desert,13.0,Deserts & Xeric Shrublands,Palearctic
3,404,acid_nox,cf_mean,0.351993,Ahklun and Kilbuck Upland Tundra,11.0,Tundra,Nearctic
4,722,acid_nox,cf_mean,0.409728,Al-Hajar foothill xeric woodlands and shrublands,8.0,"Temperate Grasslands, Savannas & Shrublands",Palearctic


In [57]:
acid_er_df["metric"] = acid_er_df["metric"].replace(rename_metrics)

In [58]:
acid_er_df["metric"].unique()

array(['leaf_mean', 'leaf_median', 'leaf_std'], dtype=object)

In [59]:
acid_er_df.to_csv(acid_er_df_fp, index=False)

In [60]:
pd.read_csv(acid_er_df_fp).head()

,ECO_ID,flow_name,metric,value,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM
0,135,acid_nox,leaf_mean,0.017107,Admiralty Islands lowland rain forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Australasia
1,785,acid_nox,leaf_mean,0.438687,Aegean and Western Turkey sclerophyllous and m...,12.0,"Mediterranean Forests, Woodlands & Scrub",Palearctic
2,807,acid_nox,leaf_mean,1.092774,Afghan Mountains semi-desert,13.0,Deserts & Xeric Shrublands,Palearctic
3,404,acid_nox,leaf_mean,0.351993,Ahklun and Kilbuck Upland Tundra,11.0,Tundra,Nearctic
4,722,acid_nox,leaf_mean,0.409728,Al-Hajar foothill xeric woodlands and shrublands,8.0,"Temperate Grasslands, Savannas & Shrublands",Palearctic


### Subcountries

In [77]:
acid_sc_fp = "../LEAFs/acidification/acidification_subcountry.gpkg"
acid_sc_df_fp = "../LEAFs/acidification/acidification_subcountry.csv"

In [78]:
inspect_gpkg(acid_sc_fp)

geometry_layer: (3422, 4), CRS=EPSG:6933, geom=['MultiPolygon'], geom_type_reported=MultiPolygon
acid_leaf_subcountry: TABLE, shape=(20532, 6), columns=['ADM1_CODE', 'flow_name', 'cf', 'cf_median', 'cf_std', '_source_file']
acid_leaf_subcountry_metadata: TABLE, shape=(6, 4), columns=['flow_name', 'impact_category', 'unit', 'source_file']


In [79]:
ac_sc_gpkg_data = load_gpkg(acid_sc_fp)

In [81]:
ac_sc_gpkg_data['acid_leaf_subcountry'] = ac_sc_gpkg_data['acid_leaf_subcountry'].rename(columns = rename_cols)

In [83]:
ac_sc_gpkg_data['acid_leaf_subcountry'].head()

,ADM1_CODE,flow_name,leaf,leaf_median,leaf_std,_source_file
0,40542,acid_nh3,0.818360,0.794068,6.332038e-02,acid_nh3.tif
1,40543,acid_nh3,0.794068,0.794068,1.110223e-16,acid_nh3.tif
2,40544,acid_nh3,0.794068,0.794068,1.110223e-16,acid_nh3.tif
3,40545,acid_nh3,0.794068,0.794068,1.110223e-16,acid_nh3.tif
4,40546,acid_nh3,1.038711,1.032421,1.680182e-02,acid_nh3.tif


In [82]:
update_gpkg_to_disk(ac_sc_gpkg_data, acid_sc_fp)

Geopackage updated into: ../LEAFs/acidification/acidification_subcountry.gpkg


#### df

In [69]:
acid_sc_df = pd.read_csv(acid_sc_df_fp)

In [70]:
acid_sc_df.head()

,ADM0_NAME,ADM1_NAME,ADM1_CODE,flow_name,metric,value
0,Burundi,Bubanza,40542,acid_nh3,leaf_mean,0.818360
1,Burundi,Bujumbura Mairie,40543,acid_nh3,leaf_mean,0.794068
2,Burundi,Bujumbura Rural,40544,acid_nh3,leaf_mean,0.794068
3,Burundi,Bururi,40545,acid_nh3,leaf_mean,0.794068
4,Burundi,Cankuzo,40546,acid_nh3,leaf_mean,1.038711


In [71]:
acid_sc_df["metric"] = acid_sc_df["metric"].replace(rename_metrics)

In [72]:
acid_sc_df["metric"].unique()

array(['leaf_mean', 'leaf_median', 'leaf_std'], dtype=object)

In [73]:
acid_sc_df.to_csv(acid_sc_df_fp, index=False)

In [74]:
pd.read_csv(acid_sc_df_fp).head()

,ADM0_NAME,ADM1_NAME,ADM1_CODE,flow_name,metric,value
0,Burundi,Bubanza,40542,acid_nh3,leaf_mean,0.818360
1,Burundi,Bujumbura Mairie,40543,acid_nh3,leaf_mean,0.794068
2,Burundi,Bujumbura Rural,40544,acid_nh3,leaf_mean,0.794068
3,Burundi,Bururi,40545,acid_nh3,leaf_mean,0.794068
4,Burundi,Cankuzo,40546,acid_nh3,leaf_mean,1.038711


### Countries

In [84]:
acid_c_fp = "../LEAFs/acidification/acidification_country.gpkg"
acid_c_df_fp = "../LEAFs/acidification/acidification_country.csv"

In [85]:
inspect_gpkg(acid_c_fp)

geometry_layer: (276, 2), CRS=EPSG:6933, geom=['MultiPolygon'], geom_type_reported=MultiPolygon
acid_leaf_country: TABLE, shape=(828, 6), columns=['ADM0_NAME', 'flow_name', 'cf', 'cf_median', 'cf_std', '_source_file']
acid_leaf_country_metadata: TABLE, shape=(3, 4), columns=['flow_name', 'impact_category', 'unit', 'source_file']


In [86]:
ac_c_gpkg_data = load_gpkg(acid_c_fp)

In [87]:
ac_c_gpkg_data['acid_leaf_country'] = ac_c_gpkg_data['acid_leaf_country'].rename(columns = rename_cols)

In [95]:
ac_c_gpkg_data['acid_leaf_country'].head()

,ADM0_NAME,flow_name,leaf,leaf_median,leaf_std,_source_file
0,Abyei,SO2,1.114701,1.114701,2.220446e-16,acid_so2.tif
1,Afghanistan,SO2,2.585822,2.334591,8.948115e-01,acid_so2.tif
2,Aksai Chin,SO2,3.324237,3.453164,5.836048e-01,acid_so2.tif
3,Albania,SO2,1.269199,1.258024,5.401168e-01,acid_so2.tif
4,Algeria,SO2,0.600109,0.583725,1.501198e-01,acid_so2.tif


In [88]:
update_gpkg_to_disk(ac_c_gpkg_data, acid_c_fp)

Geopackage updated into: ../LEAFs/acidification/acidification_country.gpkg


#### df

In [89]:
acid_c_df = pd.read_csv(acid_c_df_fp)

In [90]:
acid_c_df.head()

,ADM0_NAME,flow_name,metric,value
0,Abyei,acid_so2,cf_mean,1.114701
1,Afghanistan,acid_so2,cf_mean,2.585822
2,Aksai Chin,acid_so2,cf_mean,3.324237
3,Albania,acid_so2,cf_mean,1.269199
4,Algeria,acid_so2,cf_mean,0.600109


In [91]:
acid_c_df["metric"] = acid_c_df["metric"].replace(rename_metrics)

In [92]:
acid_c_df["metric"].unique()

array(['leaf_mean', 'leaf_median', 'leaf_std'], dtype=object)

In [93]:
acid_c_df.to_csv(acid_c_df_fp, index=False)

In [94]:
pd.read_csv(acid_c_df_fp).head()

,ADM0_NAME,flow_name,metric,value
0,Abyei,acid_so2,leaf_mean,1.114701
1,Afghanistan,acid_so2,leaf_mean,2.585822
2,Aksai Chin,acid_so2,leaf_mean,3.324237
3,Albania,acid_so2,leaf_mean,1.269199
4,Algeria,acid_so2,leaf_mean,0.600109
